# GuideModel - Quick Implementation Test

This notebook tests the `GuideModel` class from `plugins/guide-model`.
It validates:
- Model loading
- Guide ingestion & KV cache persistence
- Question answering
- Guide hot-reload
- Resource finder convenience method
- Context budget validation (guide size + reserved Q&A space)

## 1. Install Dependencies

In [ ]:
!pip install llama-cpp-python python-dotenv huggingface-hub

## 2. Setup — Paths & Imports

In [ ]:
import sys
import os

PLUGIN_PATH = "/home/codx-junior-projects/codx-junior/plugins/guide-model"
PROJECT_PATH = "/home/codx-junior-projects/codx-junior"

if PLUGIN_PATH not in sys.path:
    sys.path.insert(0, PLUGIN_PATH)
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

print(f"Plugin path: {PLUGIN_PATH}")
print(f"Plugin path exists: {os.path.exists(PLUGIN_PATH)}")

## 3. Config

In [ ]:
import os
import multiprocessing
from llama_cpp import Llama

CODX_JUNIOR_MODELS_PATH = os.environ.get(
    "CODX_JUNIOR_MODELS_PATH",
    "/home/codx-junior-projects/codx-junior/models"
)
MODEL_ID = "unsloth/Qwen3.5-0.8B-GGUF"
MODEL_NAME = "Qwen3.5-0.8B-Q8_0.gguf"
N_CTX = 262144

# How many tokens to reserve for questions + generated answers.
# The guide may only occupy: n_ctx - reserved_tokens - scaffold_tokens
RESERVED_TOKENS = int(os.environ.get("CODX_JUNIOR_RESERVED_TOKENS", "512"))

os.makedirs(CODX_JUNIOR_MODELS_PATH, exist_ok=True)

print(f"Models path     : {CODX_JUNIOR_MODELS_PATH}")
print(f"Model name      : {MODEL_NAME}")
print(f"Context size    : {N_CTX}")
print(f"Reserved tokens : {RESERVED_TOKENS}  (for Q&A)")
print(f"Max guide budget: {N_CTX - RESERVED_TOKENS} tokens  (before scaffold margin)")
print(f"Model exists    : {os.path.exists(os.path.join(CODX_JUNIOR_MODELS_PATH, MODEL_NAME))}")

## 4. GuideModel Implementation

### Context budget layout

```
┌─────────────────────────────────────────────────────┐  n_ctx
│                                                     │
│   Guide text tokens                                 │  ← max_guide_tokens
│                                                     │    = n_ctx
│─────────────────────────────────────────────────────│    - reserved_tokens
│   Prompt scaffold  ("Question: … Answer:")          │    - scaffold_tokens
│─────────────────────────────────────────────────────│
│   Reserved for question input + generated answer    │  ← reserved_tokens
└─────────────────────────────────────────────────────┘
```

Pass a larger `reserved_tokens` value when you expect longer questions or answers.

In [ ]:
import os
import pickle
import multiprocessing
from llama_cpp import Llama


# ---------------------------------------------------------------------------
# Errors
# ---------------------------------------------------------------------------

class GuideModelError(Exception):
    """Base error for GuideModel."""


class GuideTooLargeError(GuideModelError):
    """
    Raised when the guide text exceeds the allowed token budget.

    Budget formula
    --------------
    max_guide_tokens = n_ctx - reserved_tokens - scaffold_tokens

    Increase `reserved_tokens` to leave more room for Q&A;
    or shorten the guide so it fits within the budget.
    """

    def __init__(
        self,
        guide_tokens: int,
        max_guide_tokens: int,
        n_ctx: int,
        reserved_tokens: int,
        scaffold_tokens: int,
    ):
        self.guide_tokens = guide_tokens
        self.max_guide_tokens = max_guide_tokens
        self.n_ctx = n_ctx
        self.reserved_tokens = reserved_tokens
        self.scaffold_tokens = scaffold_tokens
        super().__init__(
            f"Guide is too large: {guide_tokens} tokens, "
            f"but the budget allows only {max_guide_tokens} tokens for the guide. "
            f"Budget: n_ctx({n_ctx}) - reserved_tokens({reserved_tokens}) "
            f"- scaffold({scaffold_tokens}) = {max_guide_tokens}. "
            f"Tip: shorten the guide or decrease reserved_tokens."
        )


class PromptTooLargeError(GuideModelError):
    """Raised when a question + guide combined prompt exceeds n_ctx."""

    def __init__(self, prompt_tokens: int, n_ctx: int):
        self.prompt_tokens = prompt_tokens
        self.n_ctx = n_ctx
        super().__init__(
            f"Full prompt is too large: {prompt_tokens} tokens exceeds "
            f"context size n_ctx={n_ctx}. "
            f"Tip: shorten the question or increase n_ctx."
        )


class NoGuideLoadedError(GuideModelError):
    """Raised when ask() is called before load_guide()."""

    def __init__(self):
        super().__init__("No guide loaded. Call load_guide() first.")


# ---------------------------------------------------------------------------
# GuideModel
# ---------------------------------------------------------------------------

class GuideModel:
    """
    A lightweight wrapper around llama-cpp-python that keeps a guide document
    in context (via KV-cache persistence) and answers questions against it.

    Context budget
    --------------
    The total context window (``n_ctx`` tokens) is partitioned as follows::

        ┌──────────────────────────────────────┐  n_ctx
        │  Guide text                          │  max_guide_tokens
        │──────────────────────────────────────│   = n_ctx - reserved_tokens
        │  Prompt scaffold ("Question:/Answer:")│     - _PROMPT_SCAFFOLD_TOKENS
        │──────────────────────────────────────│
        │  Reserved for question + answer      │  reserved_tokens
        └──────────────────────────────────────┘

    Parameters
    ----------
    model_path : str
        Absolute path to the .gguf model file.
    guide_id : str
        Unique identifier for this guide.  Used to name the on-disk state
        file so multiple guides can coexist without overwriting each other.
    n_ctx : int
        Total context window size (tokens).
    reserved_tokens : int
        Tokens reserved for the question input **and** the generated answer.
        The guide may occupy at most
        ``n_ctx - reserved_tokens - _PROMPT_SCAFFOLD_TOKENS`` tokens.
        Raise this value if you expect long questions or verbose answers;
        lower it to allow a larger guide.
    """

    # Extra tokens consumed by the prompt template scaffolding
    # ("\n\nQuestion: …\nAnswer:").
    _PROMPT_SCAFFOLD_TOKENS: int = 32

    def __init__(
        self,
        model_path: str,
        guide_id: str,
        n_ctx: int = 4096,
        reserved_tokens: int = 512,
    ):
        self.model_path = model_path
        self.guide_id = guide_id
        self.n_ctx = n_ctx
        self.reserved_tokens = reserved_tokens

        # State file lives next to the model, namespaced by guide_id
        model_dir = os.path.dirname(model_path)
        model_basename = os.path.basename(model_path)
        self.state_path = os.path.join(
            model_dir, f"{model_basename}.{guide_id}.guide.state"
        )

        self.guide_text: str = ""
        self.guide_token_count: int = 0
        self.llm: Llama | None = None

        self.threads = max(1, multiprocessing.cpu_count() // 2)
        self._load_model()

    # ------------------------------------------------------------------
    # Properties
    # ------------------------------------------------------------------

    @property
    def max_guide_tokens(self) -> int:
        """
        Maximum tokens the guide text may occupy.

        Formula::

            max_guide_tokens = n_ctx - reserved_tokens - _PROMPT_SCAFFOLD_TOKENS

        Adjusting ``reserved_tokens`` at construction time is the primary
        knob for trading guide size against Q&A space.
        """
        return self.n_ctx - self.reserved_tokens - self._PROMPT_SCAFFOLD_TOKENS

    # ------------------------------------------------------------------
    # Model lifecycle
    # ------------------------------------------------------------------

    def _load_model(self) -> None:
        """Initialises the llama model (CPU-only)."""
        self.llm = Llama(
            model_path=self.model_path,
            n_ctx=self.n_ctx,
            n_threads=self.threads,
            verbose=False,
        )

    # ------------------------------------------------------------------
    # Token counting & validation
    # ------------------------------------------------------------------

    def count_tokens(self, text: str) -> int:
        """Returns the number of tokens the model assigns to *text*."""
        return len(self.llm.tokenize(text.encode("utf-8"), add_bos=False))

    def validate_guide_size(self, guide_text: str) -> int:
        """
        Validates that *guide_text* fits within the token budget reserved
        for the guide portion of the context.

        The allowed guide budget is::

            max_guide_tokens = n_ctx - reserved_tokens - _PROMPT_SCAFFOLD_TOKENS

        A larger ``reserved_tokens`` value shrinks the guide budget but
        guarantees more room for questions and generated answers.

        Parameters
        ----------
        guide_text : str
            The guide document to validate.

        Returns
        -------
        int
            Number of tokens in *guide_text*.

        Raises
        ------
        GuideTooLargeError
            If the guide exceeds ``max_guide_tokens``.
        """
        token_count = self.count_tokens(guide_text)
        if token_count > self.max_guide_tokens:
            raise GuideTooLargeError(
                guide_tokens=token_count,
                max_guide_tokens=self.max_guide_tokens,
                n_ctx=self.n_ctx,
                reserved_tokens=self.reserved_tokens,
                scaffold_tokens=self._PROMPT_SCAFFOLD_TOKENS,
            )
        return token_count

    def validate_prompt_size(self, full_prompt: str) -> int:
        """
        Validates that a fully assembled prompt (guide + question scaffold)
        fits within ``n_ctx``.

        This is a final safety check called inside ``ask()`` before
        invoking the model.

        Returns
        -------
        int
            Number of tokens in *full_prompt*.

        Raises
        ------
        PromptTooLargeError
            If the prompt exceeds ``n_ctx``.
        """
        token_count = self.count_tokens(full_prompt)
        if token_count >= self.n_ctx:
            raise PromptTooLargeError(
                prompt_tokens=token_count,
                n_ctx=self.n_ctx,
            )
        return token_count

    def context_budget_info(self) -> dict:
        """
        Returns a human-readable summary of how the context window is split.

        Example output::

            {
                'n_ctx': 4096,
                'reserved_tokens': 512,       # for question + answer
                'scaffold_tokens': 32,        # prompt template overhead
                'max_guide_tokens': 3552,     # guide may not exceed this
                'guide_token_count': 284,     # current guide usage
                'guide_headroom': 3268,       # tokens still available
                'guide_id': 'my-guide',
            }
        """
        headroom = max(0, self.max_guide_tokens - self.guide_token_count)
        return {
            "n_ctx": self.n_ctx,
            "reserved_tokens": self.reserved_tokens,
            "scaffold_tokens": self._PROMPT_SCAFFOLD_TOKENS,
            "max_guide_tokens": self.max_guide_tokens,
            "guide_token_count": self.guide_token_count,
            "guide_headroom": headroom,
            "guide_id": self.guide_id,
        }

    # ------------------------------------------------------------------
    # KV-cache persistence
    # ------------------------------------------------------------------

    def _save_state(self) -> None:
        """Serialises the current LlamaState to disk via pickle."""
        state = self.llm.save_state()
        with open(self.state_path, "wb") as fh:
            pickle.dump(state, fh)
        print(f"[GuideModel:{self.guide_id}] State saved → {self.state_path}")

    def _restore_state(self) -> bool:
        """
        Attempts to restore LlamaState from disk.

        Returns True on success, False otherwise.
        """
        try:
            with open(self.state_path, "rb") as fh:
                state = pickle.load(fh)
            self.llm.load_state(state)
            print(f"[GuideModel:{self.guide_id}] State restored ← {self.state_path}")
            return True
        except Exception as exc:
            print(f"[GuideModel:{self.guide_id}] Failed to restore state: {exc}")
            return False

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def load_guide(self, guide_text: str, force_refresh: bool = False) -> None:
        """
        Loads *guide_text* into the model context with KV-cache persistence.

        The guide is validated for size **before** any model inference so
        that oversized guides are rejected cheaply.

        Parameters
        ----------
        guide_text : str
            The project guide / documentation to load.
        force_refresh : bool
            When True, ignores any cached state and reprocesses the guide.

        Raises
        ------
        GuideTooLargeError
            If *guide_text* exceeds ``max_guide_tokens``.
        """
        stripped = guide_text.strip()

        # Validate size BEFORE touching the model
        token_count = self.validate_guide_size(stripped)

        self.guide_text = stripped
        self.guide_token_count = token_count

        budget = self.context_budget_info()
        print(
            f"[GuideModel:{self.guide_id}] Context budget — "
            f"guide: {budget['guide_token_count']}/{budget['max_guide_tokens']} tokens, "
            f"reserved for Q&A: {budget['reserved_tokens']} tokens, "
            f"headroom: {budget['guide_headroom']} tokens"
        )

        if os.path.exists(self.state_path) and not force_refresh:
            print(f"[GuideModel:{self.guide_id}] Restoring cached state …")
            if self._restore_state():
                return

        print(f"[GuideModel:{self.guide_id}] Processing guide ({token_count} tokens) …")
        # Prime the KV cache with guide context (only 1 new token generated)
        self.llm(self.guide_text, max_tokens=1)
        self._save_state()

    def reload_guide(self, new_guide_text: str) -> None:
        """Updates the guide and overwrites the cached state."""
        self.load_guide(new_guide_text, force_refresh=True)

    def ask(self, question: str) -> str:
        """
        Queries the model using the pre-loaded guide context.

        Parameters
        ----------
        question : str
            The question to answer using the guide.

        Returns
        -------
        str
            The model's answer.

        Raises
        ------
        NoGuideLoadedError
            If ``load_guide()`` has not been called yet.
        PromptTooLargeError
            If the assembled prompt exceeds ``n_ctx``.
        """
        if not self.guide_text:
            raise NoGuideLoadedError()

        full_prompt = (
            f"{self.guide_text}\n\n"
            f"Question: {question}\n"
            f"Answer:"
        )

        # Final safety guard: full prompt must fit in context
        self.validate_prompt_size(full_prompt)

        output = self.llm(
            full_prompt,
            max_tokens=self.reserved_tokens - self._PROMPT_SCAFFOLD_TOKENS,
            stop=["Question:", "\n\n"],
            echo=False,
        )
        return output["choices"][0]["text"].strip()

    def find_resource(self, resource_description: str) -> str:
        """
        Convenience wrapper: finds a project resource by description.

        Parameters
        ----------
        resource_description : str
            Human-readable description of the resource to locate.
        """
        question = f"Where can I find the project resource for: {resource_description}?"
        return self.ask(question)

print("GuideModel defined ✓")

## 5. Download / Locate Model

In [ ]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id=MODEL_ID,
    filename=MODEL_NAME,
    local_dir=CODX_JUNIOR_MODELS_PATH,
    verbose=False,
)

print(f"Models available in {CODX_JUNIOR_MODELS_PATH}:")
for f in os.listdir(CODX_JUNIOR_MODELS_PATH):
    if f.endswith(".gguf"):
        size_mb = os.path.getsize(os.path.join(CODX_JUNIOR_MODELS_PATH, f)) / (1024 * 1024)
        print(f"  - {f}  ({size_mb:.1f} MB)")

## 6. Define a Sample Project Guide

In [ ]:
SAMPLE_GUIDE = """
# codx-junior Project Guide

## Project Structure

- `codx/junior/globals.py` — Global constants including CODX_JUNIOR_MODELS_PATH
- `codx/junior/guide_model.py` — ProjectResourceFinder core class
- `codx/junior/ai/` — AI provider integrations
- `codx/junior/api/` — REST API endpoints
- `codx/junior/settings.py` — Application settings and configuration
- `models/` — GGUF model files and KV cache state files
- `guides/` — Project guide documents (.md, .txt)
- `tests/` — Unit and integration tests
- `plugins/guide-model/` — GuideModel plugin source code
- `notebooks/` — Jupyter notebooks for experimentation
- `docker-compose.yml` — Docker Compose configuration
- `Dockerfile` — Docker build instructions
- `requirements.txt` — Python dependencies

## Key Modules

### Authentication
Authentication logic is located in `codx/junior/auth/auth_service.py`.
JWT token handling is in `codx/junior/auth/jwt_handler.py`.

### Database
Database models are in `codx/junior/db/models.py`.
Migrations are managed via `codx/junior/db/migrations/`.

### API
The main FastAPI app is defined in `codx/junior/api/app.py`.
Routes are split by domain under `codx/junior/api/routes/`.

### Configuration
Environment variables are loaded in `codx/junior/settings.py` using python-dotenv.
Copy `.env.example` to `.env` to configure the project locally.
""".strip()

print(f"Guide length: {len(SAMPLE_GUIDE)} characters")
print(f"Preview:\n{SAMPLE_GUIDE[:300]}...")

## 7. Initialize GuideModel

In [ ]:
import time

MODEL_PATH = os.path.join(CODX_JUNIOR_MODELS_PATH, MODEL_NAME)

t0 = time.time()
guide_model = GuideModel(
    model_path=MODEL_PATH,
    guide_id="codx-junior-sample",
    n_ctx=N_CTX,
    reserved_tokens=RESERVED_TOKENS,
)
elapsed = time.time() - t0

print(f"\nModel initialized in {elapsed:.2f}s")
print(f"Context budget: {guide_model.context_budget_info()}")

## 8. Context Budget Validation Tests

Verify that `GuideTooLargeError` is raised when the guide is too big,
and that the budget arithmetic is correct.

In [ ]:
print("=" * 60)
print("Context Budget Validation")
print("=" * 60)

# --- 1. Show budget breakdown ---
info = guide_model.context_budget_info()
print(f"""
Budget breakdown (no guide loaded yet):
  n_ctx              = {info['n_ctx']}
  reserved_tokens    = {info['reserved_tokens']}  ← space for Q&A
  scaffold_tokens    = {info['scaffold_tokens']}   ← prompt template overhead
  ─────────────────────────────────────────
  max_guide_tokens   = {info['max_guide_tokens']}  ← guide must fit here
  guide_token_count  = {info['guide_token_count']}    ← current usage
  guide_headroom     = {info['guide_headroom']}  ← tokens still available
""")

# --- 2. Validate the sample guide fits ---
print("Test 1: sample guide should fit …")
try:
    count = guide_model.validate_guide_size(SAMPLE_GUIDE)
    print(f"  ✓ Guide fits: {count} tokens (limit: {guide_model.max_guide_tokens})")
except GuideTooLargeError as e:
    print(f"  ✗ Unexpected error: {e}")

# --- 3. Validate that an oversized guide raises GuideTooLargeError ---
print("\nTest 2: oversized guide should raise GuideTooLargeError …")
HUGE_GUIDE = SAMPLE_GUIDE * 100  # force it way over budget
try:
    guide_model.validate_guide_size(HUGE_GUIDE)
    print("  ✗ ERROR — should have raised GuideTooLargeError!")
except GuideTooLargeError as e:
    print(f"  ✓ Correctly raised GuideTooLargeError:")
    print(f"      guide_tokens     = {e.guide_tokens}")
    print(f"      max_guide_tokens = {e.max_guide_tokens}")
    print(f"      reserved_tokens  = {e.reserved_tokens}")
    print(f"      n_ctx            = {e.n_ctx}")

# --- 4. Show how increasing reserved_tokens shrinks the guide budget ---
print("\nTest 3: reserved_tokens effect on max_guide_tokens …")
for rt in [256, 512, 1024, 2048]:
    gm_tmp = GuideModel.__new__(GuideModel)
    gm_tmp.n_ctx = N_CTX
    gm_tmp.reserved_tokens = rt
    max_g = gm_tmp.max_guide_tokens
    print(f"  reserved_tokens={rt:4d}  →  max_guide_tokens={max_g}")

## 9. Load the Guide (with KV Cache)

In [ ]:
t0 = time.time()
guide_model.load_guide(SAMPLE_GUIDE)
elapsed = time.time() - t0

print(f"\nGuide loaded in {elapsed:.2f}s")
print(f"State file exists: {os.path.exists(guide_model.state_path)}")

budget = guide_model.context_budget_info()
print(f"\nUpdated budget:")
print(f"  guide_token_count = {budget['guide_token_count']}")
print(f"  guide_headroom    = {budget['guide_headroom']}")

if os.path.exists(guide_model.state_path):
    state_size = os.path.getsize(guide_model.state_path) / (1024 * 1024)
    print(f"  State file size   = {state_size:.2f} MB")

## 10. Ask Questions

In [ ]:
questions = [
    "Where is the authentication module located?",
    "Where are the database models defined?",
    "Where is the main FastAPI application defined?",
    "How do I configure environment variables?",
]

for q in questions:
    t0 = time.time()
    answer = guide_model.ask(q)
    elapsed = time.time() - t0
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"   ({elapsed:.2f}s)")
    print("-" * 60)

## 11. Test `find_resource()` Convenience Method

In [ ]:
resources = [
    "JWT token handling",
    "API routes",
    "global constants",
]

for resource in resources:
    t0 = time.time()
    result = guide_model.find_resource(resource)
    elapsed = time.time() - t0
    print(f"Resource : {resource}")
    print(f"Found at : {result}")
    print(f"Time     : {elapsed:.2f}s")
    print("-" * 60)

## 12. Test KV Cache Speed (Reload from Disk)

In [ ]:
print("Simulating fresh startup (loading state from disk)...")
t0 = time.time()
guide_model.load_guide(SAMPLE_GUIDE, force_refresh=False)
elapsed = time.time() - t0
print(f"State restored from disk in: {elapsed:.2f}s ✓")

answer = guide_model.ask("Where are the unit tests?")
print(f"\nPost-restore query: {answer}")

## 13. Test Guide Hot-Reload

In [ ]:
UPDATED_GUIDE = SAMPLE_GUIDE + """

## New Module: Notifications
The notification service is located in `codx/junior/notifications/notification_service.py`.
Email templates are stored under `codx/junior/notifications/templates/`.
""".strip()

print("Performing hot-reload with updated guide...")
t0 = time.time()
guide_model.reload_guide(UPDATED_GUIDE)
elapsed = time.time() - t0
print(f"Guide reloaded in: {elapsed:.2f}s ✓")

answer = guide_model.ask("Where is the notification service?")
print(f"\nQ: Where is the notification service?")
print(f"A: {answer}")

## 14. Summary

In [ ]:
budget = guide_model.context_budget_info()

print("=" * 60)
print("GuideModel — Test Summary")
print("=" * 60)
print(f"  guide_id           : {budget['guide_id']}")
print(f"  model              : {MODEL_NAME}")
print(f"  n_ctx              : {budget['n_ctx']}")
print(f"  reserved_tokens    : {budget['reserved_tokens']}  ← for Q&A")
print(f"  scaffold_tokens    : {budget['scaffold_tokens']}")
print(f"  max_guide_tokens   : {budget['max_guide_tokens']}")
print(f"  guide_token_count  : {budget['guide_token_count']}")
print(f"  guide_headroom     : {budget['guide_headroom']}")
print(f"  CPU threads        : {guide_model.threads}")
print(f"  state_path         : {guide_model.state_path}")

if os.path.exists(guide_model.state_path):
    size = os.path.getsize(guide_model.state_path) / (1024 * 1024)
    print(f"  state_size         : {size:.2f} MB")

print("=" * 60)
print("All tests passed ✓")